# Fake News Detection with Graph Neural Networks
Date: 2026

In [1]:
import os
import sys
from torch.utils.data import DataLoader, ConcatDataset, WeightedRandomSampler, Subset
import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.data import Batch
from torch_geometric.utils import dropout_edge
from pathlib import Path
root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_dir not in sys.path:
    sys.path.append(root_dir)
from src.models.Fakenews_GNN import Fakenews_GNN
from src.utils.data_loader import FNNDataset, add_bot_score_feature
import warnings
warnings.filterwarnings("ignore")
import numpy as np

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

data_dir = Path("../data/raw/")
gos_map_path = data_dir / "gos_id_twitter_mapping.pkl"
root = os.path.join(root_dir, "data")

dataset_gos = FNNDataset(root=root, name="gossipcop", feature="profile")
add_bot_score_feature(dataset_gos, gos_map_path, "./scores_gos.json")

pol_map_path = data_dir / "pol_id_twitter_mapping.pkl"
dataset_pol = FNNDataset(root=root, name="politifact", feature="profile")
add_bot_score_feature(dataset_pol, pol_map_path, "./scores_pol.json")

train_dataset_gos = Subset(dataset_gos, dataset_gos.train_idx)
val_dataset_gos   = Subset(dataset_gos, dataset_gos.test_idx)
train_dataset_gos = ConcatDataset([train_dataset_gos, val_dataset_gos])
test_dataset_gos  = Subset(dataset_gos, dataset_gos.val_idx)

train_dataset_pol = Subset(dataset_pol, dataset_pol.train_idx)
val_dataset_pol   = Subset(dataset_pol, dataset_pol.test_idx)
train_dataset_pol = ConcatDataset([train_dataset_pol, val_dataset_pol])
test_dataset_pol  = Subset(dataset_pol, dataset_pol.val_idx)

print(len(train_dataset_gos))
print(len(train_dataset_pol))

train_dataset = ConcatDataset([train_dataset_gos, train_dataset_pol])
len_gos = len(train_dataset_gos)
len_pol = len(train_dataset_pol)
weights_gos = torch.ones(len_gos) / len_gos
weights_pol = torch.ones(len_pol) / len_pol
sample_weights = torch.cat([weights_gos, weights_pol])
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

GPU KNN imputation: 100%|██████████████████████████████████████████████████████████████| 37/37 [00:00<00:00, 577.82it/s]

4372
252


In [6]:
best_set1 = {"batch_size": 16, "conv_type": "GraphSAGE", "dropout": 0.368, "hidden_channels": 128 ,"lr": 0.0052, 
             "num_layers":4, "optimizer": "AdamW", "pooling": "mean", "use_batchnorm": True, "weight_decay": 0.0024}
best_set2 = {"batch_size": 16, "conv_type": "GAT", "dropout": 0.17, "hidden_channels": 128,  "lr": 0.0057, 
             "num_layers":2, "optimizer": "AdamW", "pooling": "max", "use_batchnorm": False, "weight_decay": 0.0098}
best_set3 = {"batch_size": 32, "conv_type": "GAT", "dropout": 0.07, "hidden_channels": 32,  "lr":0.0045, 
             "num_layers":3, "optimizer": "Adam", "pooling": "max", "use_batchnorm": True, "weight_decay": 0.0051
}


# -------------------------------------------------------
# Validation / Testing function
# -------------------------------------------------------

def evaluate(loader):
    correct, total, loss_sum = 0, 0, 0
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.batch)
            loss_sum += F.cross_entropy(out, data.y).item()
            pred = out.argmax(dim=1)
            correct += (pred == data.y).sum().item()
            total += data.y.size(0)
    return loss_sum / len(loader), correct / total

def mask_features(x, mask_prob):
    if mask_prob <= 0:
        return x
    mask = torch.rand_like(x) < mask_prob
    return x * (~mask)

def apply_edge_dropout(edge_index, drop_prob):
    if drop_prob <= 0:
        return edge_index
    edge_index, _ = dropout_edge(edge_index, p=drop_prob)
    return edge_index

In [4]:
set_n = 1
for param_set in [best_set1, best_set2, best_set3]:
    model = Fakenews_GNN(
        in_channels=11,
        hidden_channels=param_set["hidden_channels"],
        out_channels=2,
        num_layers=param_set["num_layers"],
        conv_type=param_set["conv_type"],
        dropout=param_set["dropout"],
        use_batchnorm=param_set["use_batchnorm"],
        pooling=param_set["pooling"]
    ).to(device)
    
    optimizer_cls = torch.optim.Adam if param_set["optimizer"] == "Adam" else torch.optim.AdamW
    optimizer = optimizer_cls(model.parameters(), lr=param_set["lr"], weight_decay=param_set["weight_decay"])
    
    train_loader = PyGDataLoader(
        train_dataset,
        batch_size=param_set["batch_size"],
        sampler=sampler
    )
    
    test_loader_gos = PyGDataLoader(test_dataset_gos, batch_size=param_set["batch_size"], shuffle=False)
    test_loader_pol = PyGDataLoader(test_dataset_pol, batch_size=param_set["batch_size"], shuffle=False)
    
    feature_mask_prob = 0.05
    edge_drop_prob =  0.1
    
    for epoch in range(100):
        model.train()
        total_loss = 0
    
        for data in train_loader:
            data = data.to(device)
    
            # --------------------------------------
            # Apply masking augmentations
            # --------------------------------------
            data.x = mask_features(data.x, mask_prob=feature_mask_prob)
            data.edge_index = apply_edge_dropout(data.edge_index, drop_prob=edge_drop_prob)
    
            optimizer.zero_grad()
            out = model(data.x, data.edge_index, data.batch)
            loss = F.cross_entropy(out, data.y)
            loss.backward()
            optimizer.step()
    
            total_loss += loss.item()
    
        train_loss = total_loss / len(train_loader)
    
        # --- Validation on both datasets ---
        val_loss_gos, val_acc_gos = evaluate(test_loader_gos)
        val_loss_pol, val_acc_pol = evaluate(test_loader_pol)
    
        print(
            f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, "
            f"Test GossipCop Acc: {val_acc_gos:.4f}, Test Politifact Acc: {val_acc_pol:.4f}"
        )
    
    
    # -------------------------------------------------------
    # Final Tests
    # -------------------------------------------------------
    
    test_loss_gos, test_acc_gos = evaluate(test_loader_gos)
    test_loss_pol, test_acc_pol = evaluate(test_loader_pol)
    
    print(f"Test GossipCop: Loss {test_loss_gos:.4f}, Acc {test_acc_gos:.4f}")
    print(f"Test Politifact: Loss {test_loss_pol:.4f}, Acc {test_acc_pol:.4f}")
    print(f"saving as GNN_Set_Pol_{set_n}")
    torch.save(model.state_dict(), f"GNN_Set_Pol_{set_n}.pt")
    set_n += 1

Epoch 1, Train Loss: 0.5731, Test GossipCop Acc: 0.7949, Test Politifact Acc: 0.6613
Epoch 2, Train Loss: 0.5015, Test GossipCop Acc: 0.5833, Test Politifact Acc: 0.6774
Epoch 3, Train Loss: 0.4612, Test GossipCop Acc: 0.7253, Test Politifact Acc: 0.7097
Epoch 4, Train Loss: 0.4286, Test GossipCop Acc: 0.7143, Test Politifact Acc: 0.7419
Epoch 5, Train Loss: 0.4141, Test GossipCop Acc: 0.7463, Test Politifact Acc: 0.7419
Epoch 6, Train Loss: 0.3869, Test GossipCop Acc: 0.8233, Test Politifact Acc: 0.7097
Epoch 7, Train Loss: 0.3960, Test GossipCop Acc: 0.8306, Test Politifact Acc: 0.7258
Epoch 8, Train Loss: 0.3621, Test GossipCop Acc: 0.8132, Test Politifact Acc: 0.7581
Epoch 9, Train Loss: 0.3427, Test GossipCop Acc: 0.7811, Test Politifact Acc: 0.7903
Epoch 10, Train Loss: 0.3351, Test GossipCop Acc: 0.7885, Test Politifact Acc: 0.8226
Epoch 11, Train Loss: 0.3098, Test GossipCop Acc: 0.8672, Test Politifact Acc: 0.7097
Epoch 12, Train Loss: 0.3241, Test GossipCop Acc: 0.8516, Test 

In [24]:
from src.utils.data_loader import preprocess_bot_score_edges

train_dataset = ConcatDataset([train_dataset_gos, train_dataset_pol])

train_dataset = preprocess_bot_score_edges(train_dataset, diff_threshold=0.4)

param_set = best_set2

model = Fakenews_GNN(
    in_channels=11,
    hidden_channels=param_set["hidden_channels"],
    out_channels=2,
    num_layers=param_set["num_layers"],
    conv_type=param_set["conv_type"],
    dropout=param_set["dropout"],
    use_batchnorm=param_set["use_batchnorm"],
    pooling=param_set["pooling"]
).to(device)
    
optimizer_cls = torch.optim.Adam if param_set["optimizer"] == "Adam" else torch.optim.AdamW
optimizer = optimizer_cls(model.parameters(), lr=param_set["lr"], weight_decay=param_set["weight_decay"])
    
train_loader = PyGDataLoader(
    train_dataset,
    batch_size=param_set["batch_size"],
    sampler=sampler
)
    
test_loader_gos = PyGDataLoader(test_dataset_gos, batch_size=param_set["batch_size"], shuffle=False)
test_loader_pol = PyGDataLoader(test_dataset_pol, batch_size=param_set["batch_size"], shuffle=False)
    
feature_mask_prob = 0.05
edge_drop_prob =  0.1
    
for epoch in range(100):
    model.train()
    total_loss = 0
    
    for data in train_loader:
        data = data.to(device)

        data.x = mask_features(data.x, mask_prob=feature_mask_prob)
        data.edge_index = apply_edge_dropout(data.edge_index, drop_prob=edge_drop_prob)
    
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.batch)
        loss = F.cross_entropy(out, data.y)
        loss.backward()
        optimizer.step()
    
        total_loss += loss.item()
    
    train_loss = total_loss / len(train_loader)
    
        # --- Validation on both datasets ---
    val_loss_gos, val_acc_gos = evaluate(test_loader_gos)
    val_loss_pol, val_acc_pol = evaluate(test_loader_pol)
    
    print(
        f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, "
        f"Test GossipCop Acc: {val_acc_gos:.4f}, Test Politifact Acc: {val_acc_pol:.4f}"
    )
    
    
# -------------------------------------------------------
# Final Tests
# -------------------------------------------------------
    
test_loss_gos, test_acc_gos = evaluate(test_loader_gos)
test_loss_pol, test_acc_pol = evaluate(test_loader_pol)
    
print(f"Test GossipCop: Loss {test_loss_gos:.4f}, Acc {test_acc_gos:.4f}")
print(f"Test Politifact: Loss {test_loss_pol:.4f}, Acc {test_acc_pol:.4f}")

Epoch 1, Train Loss: 0.5484, Test GossipCop Acc: 0.8498, Test Politifact Acc: 0.6774
Epoch 2, Train Loss: 0.4306, Test GossipCop Acc: 0.8864, Test Politifact Acc: 0.7903
Epoch 3, Train Loss: 0.3879, Test GossipCop Acc: 0.9011, Test Politifact Acc: 0.7419
Epoch 4, Train Loss: 0.3527, Test GossipCop Acc: 0.9277, Test Politifact Acc: 0.7742
Epoch 5, Train Loss: 0.3049, Test GossipCop Acc: 0.9441, Test Politifact Acc: 0.7258
Epoch 6, Train Loss: 0.3018, Test GossipCop Acc: 0.9515, Test Politifact Acc: 0.7903
Epoch 7, Train Loss: 0.2724, Test GossipCop Acc: 0.9478, Test Politifact Acc: 0.7581
Epoch 8, Train Loss: 0.2559, Test GossipCop Acc: 0.9414, Test Politifact Acc: 0.7097
Epoch 9, Train Loss: 0.2509, Test GossipCop Acc: 0.9121, Test Politifact Acc: 0.7097
Epoch 10, Train Loss: 0.2413, Test GossipCop Acc: 0.9176, Test Politifact Acc: 0.7742
Epoch 11, Train Loss: 0.2400, Test GossipCop Acc: 0.9304, Test Politifact Acc: 0.7581
Epoch 12, Train Loss: 0.2320, Test GossipCop Acc: 0.9625, Test 

In [ ]:
0.4 is best :)